# EchoFactory - FAN: Data Exploration (EDA)
**Dataset**: `/kaggle/input/datasets/bisheshgiri/mimii-dataset`
**Machine Target**: `fan` | **Model**: STgram-MFN


In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

print(f'librosa {librosa.__version__} | numpy {np.__version__}')


In [ ]:
DATASET_ROOT = '/kaggle/input/datasets/bisheshgiri/mimii-dataset'
TARGET_SNR = '0_dB'
MACHINE_TYPE = 'fan'
SAMPLE_RATE = 16000

print('Isi folder dataset:')
if os.path.exists(DATASET_ROOT):
    for f in sorted(os.listdir(DATASET_ROOT)):
        print(f'  {f}')
else:
    print(f'⚠️ Path {DATASET_ROOT} tidak ditemukan di environment ini (Aman jika running di Kaggle).')


In [ ]:
mpath = os.path.join(DATASET_ROOT, f'{TARGET_SNR}_{MACHINE_TYPE}', MACHINE_TYPE)
rows = []
if os.path.exists(mpath):
    for mid in sorted(os.listdir(mpath)):
        for cond in ['normal', 'abnormal']:
            cp = os.path.join(mpath, mid, cond)
            if not os.path.exists(cp): continue
            n = len(glob.glob(os.path.join(cp, '*.wav')))
            rows.append({'machine': MACHINE_TYPE, 'machine_id': mid, 'condition': cond, 'n_files': n})

df = pd.DataFrame(rows)
if not df.empty:
    print(df.groupby(['machine_id', 'condition'])['n_files'].sum().unstack(fill_value=0))
    print(f'Total file {MACHINE_TYPE} (0dB): {df["n_files"].sum()}')


In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    pv = df.groupby(['machine_id', 'condition'])['n_files'].sum().unstack(fill_value=0)
    colors = ['#2ecc71' if c == 'normal' else '#e74c3c' for c in pv.columns]
    pv.plot(kind='bar', ax=ax, color=colors, width=0.6, edgecolor='white')
    ax.set_title(f'Distribusi File {{MACHINE_TYPE.upper()}} (SNR 0dB)', fontweight='bold')
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(title='Kondisi')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/file_distribution_{MACHINE_TYPE}.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
def get_sample(cond='normal', mid='id_00'):
    base = os.path.join(DATASET_ROOT, f'{TARGET_SNR}_{MACHINE_TYPE}', MACHINE_TYPE, mid, cond)
    files = sorted(glob.glob(os.path.join(base, '*.wav')))
    return files[0] if files else None

path = get_sample('normal')
if path:
    wav, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    print(f'Loaded: {os.path.basename(path)}')
    print(f'Durasi: {len(wav)/sr:.2f}s | SR: {sr}Hz | Shape: {wav.shape}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(f'Waveform {{MACHINE_TYPE.upper()}}: Normal vs Abnormal (0dB)', fontsize=14, fontweight='bold')
for col, cond in enumerate(['normal', 'abnormal']):
    p = get_sample(cond)
    if not p: continue
    w, sr_ = librosa.load(p, sr=SAMPLE_RATE, mono=True)
    t = np.linspace(0, len(w)/sr_, len(w))
    color = '#2ecc71' if cond == 'normal' else '#e74c3c'
    ax = axes[col]
    ax.plot(t, w, color=color, lw=0.4, alpha=0.9)
    ax.set_title(f'{cond.upper()}', fontweight='bold')
    ax.set_xlabel('Waktu (s)')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'/kaggle/working/waveform_{MACHINE_TYPE}.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Log-Mel Spectrogram {{MACHINE_TYPE.upper()}}: Normal vs Abnormal', fontsize=14, fontweight='bold')
for col, cond in enumerate(['normal', 'abnormal']):
    p = get_sample(cond)
    if not p: continue
    w, sr_ = librosa.load(p, sr=SAMPLE_RATE, mono=True)
    mel = librosa.feature.melspectrogram(y=w, sr=sr_, n_mels=128, n_fft=1024, hop_length=512)
    spec = librosa.power_to_db(mel, ref=np.max)
    ax = axes[col]
    img = librosa.display.specshow(spec, sr=sr_, hop_length=512, x_axis='time', y_axis='mel', ax=ax, cmap='magma')
    ax.set_title(f'{cond.upper()}', fontweight='bold')
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
plt.tight_layout()
plt.savefig(f'/kaggle/working/mel_spec_{MACHINE_TYPE}.png', dpi=150, bbox_inches='tight')
plt.show()
